# Processamento de Língua Natural - EP

## BERTimbau puro com classificação multiclasse

Este notebook é um experimento de controle.

A ideia é testar o BERTimbau sem os componentes adicionais usados na versão anterior. Neste experimento não há:

- classificação ordinal;
- características estruturais manuais;
- TAPT;
- TF-IDF.

O modelo recebe apenas o texto e aprende diretamente as três classes `c1`, `c234` e `c5`.

A arquitetura é:

`texto → BERTimbau → vetor [CLS] → dropout → camada linear → 3 classes`

O objetivo é verificar se o BERTimbau, sozinho, consegue superar o baseline antes de acrescentarmos uma estratégia mais elaborada para a versão final.

## Bibliotecas

Caso seja necessário instalar as dependências no Colab:

```python
%pip install pandas openpyxl numpy scikit-learn torch transformers accelerate tqdm
```

In [ ]:
import gc
import random
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold

from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

## Configuração do experimento

Mantemos a configuração curta usada nos últimos testes para facilitar a comparação:

- 3 folds;
- 2 épocas;
- até 128 tokens por texto.

Os resultados principais são gravados em `resultados_bert_puro.txt`.

In [ ]:
SEED = 42

MODELO_BASE = "neuralmind/bert-base-portuguese-cased"

N_SPLITS = 3
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

LOG_PATH = "resultados_bert_puro.txt"


def fixar_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def log_resultado(mensagem=""):
    mensagem = str(mensagem)
    print(mensagem)

    with open(LOG_PATH, "a", encoding="utf-8") as arquivo:
        arquivo.write(mensagem + "\n")


fixar_seed(SEED)

with open(LOG_PATH, "w", encoding="utf-8") as arquivo:
    arquivo.write("EP1 - BERTimbau puro multiclasse\n")
    arquivo.write("=" * 60 + "\n")
    arquivo.write(
        f"Início: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    )
    arquivo.write(f"Modelo base: {MODELO_BASE}\n")
    arquivo.write(f"Dispositivo: {DEVICE}\n")
    arquivo.write(
        f"Configuração: {N_SPLITS} folds, {EPOCHS} épocas, "
        f"MAX_LENGTH={MAX_LENGTH}, BATCH_SIZE={BATCH_SIZE}\n"
    )
    arquivo.write(f"Learning rate: {LEARNING_RATE}\n")
    arquivo.write(f"Dropout: {DROPOUT}\n")
    arquivo.write("=" * 60 + "\n\n")

log_resultado(f"Dispositivo: {DEVICE}")
log_resultado(
    f"Configuração: {N_SPLITS} folds, {EPOCHS} épocas, "
    f"máximo de {MAX_LENGTH} tokens"
)
log_resultado(f"Log: {LOG_PATH}")

## Importação dos dados

O texto é mantido praticamente como veio da planilha. Como o BERTimbau é *cased*, não transformamos tudo em minúsculas.

In [ ]:
dados_treino = pd.read_excel(
    "dados/train.xlsx",
    converters={"resp_text": str, "clarity": str},
)

dados_teste = pd.read_excel(
    "dados/test1.xlsx",
    converters={"resp_text": str, "clarity": str},
)

X_treino = (
    dados_treino["resp_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

Y_treino = (
    dados_treino["clarity"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_teste = (
    dados_teste["resp_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

log_resultado()
log_resultado(f"Instâncias de treino: {len(X_treino)}")
log_resultado(f"Instâncias de teste: {len(X_teste)}")
log_resultado("Distribuição das classes no treino:")
log_resultado(dados_treino["clarity"].value_counts().to_string())

## Pré-processamento e rótulos

O pré-processamento apenas normaliza espaços em branco. Não removemos stopwords, não aplicamos stemming e não substituímos palavras por tokens artificiais.

As classes são convertidas para índices apenas para o treinamento da rede.

In [ ]:
def preprocess_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


LABEL_TO_INDEX = {
    "c1": 0,
    "c234": 1,
    "c5": 2,
}

INDEX_TO_LABEL = {
    0: "c1",
    1: "c234",
    2: "c5",
}

labels_invalidos = sorted(
    set(Y_treino) - set(LABEL_TO_INDEX)
)

if labels_invalidos:
    raise ValueError(
        f"Rótulos inesperados no treino: {labels_invalidos}"
    )

y_indices = np.array(
    [LABEL_TO_INDEX[label] for label in Y_treino],
    dtype=np.int64,
)

## Dataset e padding dinâmico

O padding é feito por batch. Assim, cada batch é preenchido apenas até o maior texto presente nele, evitando processamento desnecessário.

In [ ]:
class ClarezaDataset(Dataset):
    def __init__(
        self,
        textos,
        tokenizer,
        labels=None,
        max_length=128,
    ):
        textos = [
            preprocess_text(texto)
            for texto in textos
        ]

        self.tokens = tokenizer(
            textos,
            padding=False,
            truncation=True,
            max_length=max_length,
            add_special_tokens=True,
        )

        self.labels = None

        if labels is not None:
            self.labels = torch.tensor(
                labels,
                dtype=torch.long,
            )

    def __len__(self):
        return len(self.tokens["input_ids"])

    def __getitem__(self, idx):
        item = {
            key: self.tokens[key][idx]
            for key in self.tokens
        }

        if self.labels is not None:
            item["labels"] = self.labels[idx]

        return item


def criar_collate_fn(tokenizer):
    def collate_fn(batch):
        token_batch = []

        for item in batch:
            token_item = {
                key: item[key]
                for key in item
                if key != "labels"
            }
            token_batch.append(token_item)

        padded = tokenizer.pad(
            token_batch,
            padding=True,
            return_tensors="pt",
        )

        if "labels" in batch[0]:
            padded["labels"] = torch.stack(
                [item["labels"] for item in batch]
            )

        return padded

    return collate_fn

## Modelo

O encoder é o BERTimbau. Usamos o vetor do token `[CLS]` como representação do documento e uma camada linear para produzir os três logits da classificação.

Não há características extras nem estrutura ordinal.

In [ ]:
class BertMulticlasse(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes=3,
        dropout=0.10,
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            hidden_size,
            num_classes,
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)

        logits = self.classifier(cls)

        return logits

## Treinamento e predição

O treinamento usa `CrossEntropyLoss`, que é a loss padrão para classificação multiclasse.

O mesmo learning rate é usado em todo o modelo para manter este experimento como um controle simples.

In [ ]:
def treinar_modelo(
    model,
    dataset,
    tokenizer,
    epochs,
):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=criar_collate_fn(tokenizer),
        pin_memory=PIN_MEMORY,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(loader) * epochs
    warmup_steps = int(total_steps * 0.10)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    criterion = nn.CrossEntropyLoss()

    model.to(DEVICE)

    historico_loss = []

    for epoch in range(epochs):
        model.train()
        losses = []

        barra = tqdm(
            loader,
            desc=f"Época {epoch + 1}/{epochs}",
            leave=True,
        )

        for batch in barra:
            optimizer.zero_grad()

            labels = batch["labels"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            input_ids = batch["input_ids"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            attention_mask = batch["attention_mask"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            token_type_ids = batch.get(
                "token_type_ids"
            )

            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()
            scheduler.step()

            losses.append(loss.item())

            barra.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        media_loss = float(
            np.mean(losses)
        )

        historico_loss.append(
            media_loss
        )

        log_resultado(
            f"Época {epoch + 1}/{epochs} "
            f"- loss média: {media_loss:.4f}"
        )

    return historico_loss


@torch.inference_mode()
def prever_indices(
    model,
    dataset,
    tokenizer,
    descricao="Predição",
):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=criar_collate_fn(tokenizer),
        pin_memory=PIN_MEMORY,
    )

    model.eval()
    model.to(DEVICE)

    predicoes = []

    for batch in tqdm(
        loader,
        desc=descricao,
        leave=False,
    ):
        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        token_type_ids = batch.get(
            "token_type_ids"
        )

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        pred = torch.argmax(
            logits,
            dim=1,
        )

        predicoes.extend(
            pred.cpu().numpy().tolist()
        )

    return np.array(
        predicoes,
        dtype=np.int64,
    )

## Validação cruzada

Mantemos a validação estratificada. Além da média de acurácia, o notebook salva as previsões *out-of-fold* para gerar uma matriz de confusão geral.

Isso ajuda a identificar quais classes o BERT mais confunde.

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

resultados = []

predicoes_oof = np.full(
    len(X_treino),
    -1,
    dtype=np.int64,
)

inicio_cv = time.time()

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_treino, y_indices),
    start=1,
):
    fixar_seed(SEED + fold)

    inicio_fold = time.time()

    log_resultado()
    log_resultado("=" * 60)
    log_resultado(f"FOLD {fold}/{N_SPLITS}")
    log_resultado("=" * 60)

    textos_fold_treino = [
        X_treino[i]
        for i in train_idx
    ]

    textos_fold_validacao = [
        X_treino[i]
        for i in val_idx
    ]

    y_fold_treino = y_indices[train_idx]
    y_fold_validacao = y_indices[val_idx]

    tokenizer = AutoTokenizer.from_pretrained(
        MODELO_BASE
    )

    dataset_fold_treino = ClarezaDataset(
        textos=textos_fold_treino,
        tokenizer=tokenizer,
        labels=y_fold_treino,
        max_length=MAX_LENGTH,
    )

    dataset_fold_validacao = ClarezaDataset(
        textos=textos_fold_validacao,
        tokenizer=tokenizer,
        labels=y_fold_validacao,
        max_length=MAX_LENGTH,
    )

    model = BertMulticlasse(
        model_name=MODELO_BASE,
        num_classes=3,
        dropout=DROPOUT,
    )

    treinar_modelo(
        model=model,
        dataset=dataset_fold_treino,
        tokenizer=tokenizer,
        epochs=EPOCHS,
    )

    pred_treino = prever_indices(
        model,
        dataset_fold_treino,
        tokenizer,
        descricao=f"Fold {fold} - treino",
    )

    pred_validacao = prever_indices(
        model,
        dataset_fold_validacao,
        tokenizer,
        descricao=f"Fold {fold} - validação",
    )

    predicoes_oof[val_idx] = pred_validacao

    acc_treino = accuracy_score(
        y_fold_treino,
        pred_treino,
    )

    acc_validacao = accuracy_score(
        y_fold_validacao,
        pred_validacao,
    )

    tempo_fold = time.time() - inicio_fold

    resultados.append(
        {
            "fold": fold,
            "train_accuracy": acc_treino,
            "validation_accuracy": acc_validacao,
            "tempo_segundos": tempo_fold,
        }
    )

    log_resultado(
        f"Acurácia de treino: "
        f"{acc_treino:.2%}"
    )

    log_resultado(
        f"Acurácia de validação: "
        f"{acc_validacao:.2%}"
    )

    log_resultado(
        f"Tempo do fold: "
        f"{tempo_fold / 60:.2f} min"
    )

    del model
    del tokenizer
    del dataset_fold_treino
    del dataset_fold_validacao

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

resultados_df = pd.DataFrame(
    resultados
)

acuracia_validacao = (
    resultados_df[
        "validation_accuracy"
    ].mean()
)

desvio_validacao = (
    resultados_df[
        "validation_accuracy"
    ].std()
)

acuracia_treino_cv = (
    resultados_df[
        "train_accuracy"
    ].mean()
)

gap = (
    acuracia_treino_cv
    - acuracia_validacao
)

tempo_cv = (
    time.time() - inicio_cv
)

log_resultado()
log_resultado("=" * 60)
log_resultado("RESUMO DA VALIDAÇÃO CRUZADA")
log_resultado("=" * 60)

log_resultado(
    resultados_df.to_string(
        index=False
    )
)

log_resultado()

log_resultado(
    f"Acurácia Média da Validação Cruzada: "
    f"{acuracia_validacao:.2%}"
)

log_resultado(
    f"Desvio-padrão da Validação: "
    f"{desvio_validacao:.2%}"
)

log_resultado(
    f"Acurácia Média de Treino nos Folds: "
    f"{acuracia_treino_cv:.2%}"
)

log_resultado(
    f"Gap Treino-Validação: "
    f"{gap:.2%}"
)

log_resultado(
    f"Tempo total da validação cruzada: "
    f"{tempo_cv / 60:.2f} min"
)

## Análise das previsões de validação

A matriz de confusão abaixo usa apenas previsões feitas em exemplos que estavam fora do fold de treinamento correspondente.

In [ ]:
ordem_classes = [0, 1, 2]
nomes_classes = ["c1", "c234", "c5"]

matriz = confusion_matrix(
    y_indices,
    predicoes_oof,
    labels=ordem_classes,
)

relatorio = classification_report(
    y_indices,
    predicoes_oof,
    labels=ordem_classes,
    target_names=nomes_classes,
    digits=4,
    zero_division=0,
)

matriz_df = pd.DataFrame(
    matriz,
    index=[
        f"real_{classe}"
        for classe in nomes_classes
    ],
    columns=[
        f"pred_{classe}"
        for classe in nomes_classes
    ],
)

log_resultado()
log_resultado("=" * 60)
log_resultado("MATRIZ DE CONFUSÃO OOF")
log_resultado("=" * 60)
log_resultado(matriz_df.to_string())

log_resultado()
log_resultado("=" * 60)
log_resultado("CLASSIFICATION REPORT OOF")
log_resultado("=" * 60)
log_resultado(relatorio)

matriz_df

## Treino do modelo final

Depois da validação cruzada, o BERT é treinado novamente usando todas as instâncias rotuladas.

In [ ]:
fixar_seed(SEED)

inicio_treino_final = time.time()

tokenizer_final = AutoTokenizer.from_pretrained(
    MODELO_BASE
)

dataset_treino_final = ClarezaDataset(
    textos=X_treino,
    tokenizer=tokenizer_final,
    labels=y_indices,
    max_length=MAX_LENGTH,
)

model_final = BertMulticlasse(
    model_name=MODELO_BASE,
    num_classes=3,
    dropout=DROPOUT,
)

treinar_modelo(
    model=model_final,
    dataset=dataset_treino_final,
    tokenizer=tokenizer_final,
    epochs=EPOCHS,
)

pred_treino_final = prever_indices(
    model_final,
    dataset_treino_final,
    tokenizer_final,
    descricao="Treino final - avaliação",
)

acc_treino_final = accuracy_score(
    y_indices,
    pred_treino_final,
)

tempo_treino_final = (
    time.time() - inicio_treino_final
)

log_resultado()
log_resultado("=" * 60)
log_resultado("TREINO FINAL")
log_resultado("=" * 60)

log_resultado(
    f"Acurácia do Treino (modelo final): "
    f"{acc_treino_final:.2%}"
)

log_resultado(
    f"Tempo do treino final: "
    f"{tempo_treino_final / 60:.2f} min"
)

## Execução do teste e salvamento

A ordem das linhas de `test1.xlsx` é preservada. O arquivo final continua com apenas as colunas `resp_text` e `clarity`.

In [ ]:
dataset_teste = ClarezaDataset(
    textos=X_teste,
    tokenizer=tokenizer_final,
    labels=None,
    max_length=MAX_LENGTH,
)

pred_teste_indices = prever_indices(
    model_final,
    dataset_teste,
    tokenizer_final,
    descricao="Teste",
)

Y_previsto = [
    INDEX_TO_LABEL[int(indice)]
    for indice in pred_teste_indices
]

assert len(Y_previsto) == len(X_teste)

assert set(Y_previsto).issubset(
    {"c1", "c234", "c5"}
)

distribuicao_teste = (
    pd.Series(Y_previsto)
    .value_counts()
)

log_resultado()
log_resultado("=" * 60)
log_resultado(
    "DISTRIBUIÇÃO DAS PREVISÕES DO TESTE"
)
log_resultado("=" * 60)
log_resultado(
    distribuicao_teste.to_string()
)

distribuicao_teste

In [ ]:
teste_rotulado = pd.DataFrame(
    {
        "resp_text": X_teste,
        "clarity": Y_previsto,
    }
)

teste_rotulado.to_excel(
    "teste_rotulado_bert_puro.xlsx",
    index=False,
)

oof_df = pd.DataFrame(
    {
        "resp_text": X_treino,
        "clarity_real": Y_treino,
        "clarity_pred_oof": [
            INDEX_TO_LABEL[int(indice)]
            for indice in predicoes_oof
        ],
    }
)

oof_df.to_csv(
    "predicoes_oof_bert_puro.csv",
    index=False,
    encoding="utf-8",
)

log_resultado()
log_resultado(
    "Arquivo de teste salvo: "
    "teste_rotulado_bert_puro.xlsx"
)

log_resultado(
    "Predições OOF salvas: "
    "predicoes_oof_bert_puro.csv"
)

log_resultado(
    f"Arquivo de log salvo: {LOG_PATH}"
)

log_resultado(
    f"Fim: "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

teste_rotulado.head()

## Log do experimento

A célula abaixo exibe o mesmo conteúdo salvo no arquivo `resultados_bert_puro.txt`.

In [ ]:
with open(
    LOG_PATH,
    "r",
    encoding="utf-8",
) as arquivo:
    print(arquivo.read())

## Próximo passo

Este notebook é apenas o controle com BERT multiclasse.

A versão final do EP não precisa ficar limitada a esta arquitetura. Se o BERT puro mostrar potencial, o próximo passo será acrescentar uma estratégia que tenha justificativa específica para o problema e contribua para o critério de método inovador / bem elaborado.